# GoMeal ML Scopes And Diversity

This notebook documents two feed-quality layers:

- `routes/feed/scopes/scope.py`: creates scope tags like `dessert`, `soup`, `quick`, `high_protein`
- `routes/feed/rank/diversify.py`: reranks candidate posts to reduce near-duplicates

These are not the main learning system, but they make ranking feel cleaner and more intentional.

## Scope Creation Flow

`routes/embed` builds post text from:

- dish name
- description
- difficulty
- ingredients
- steps
- nutrition
- dietary data

Then it calls `_create_scope(_text)` and stores `scope_tags` in `post_embeddings`.

```text
post row
  -> _build_post_text
  -> generate embedding
  -> _create_scope
  -> upsert post_embeddings
```

In [ ]:
import re


FEED_SCOPE_REGISTRY = {
    "dessert": {
        "family": "intent",
        "strong_keywords": ["dessert", "cake", "cookie", "brownie", "pie"],
        "weak_keywords": ["chocolate", "sweet", "frosting"],
        "negative_keywords": ["chicken", "beef", "salmon", "main course"],
        "threshold": 2,
    },
    "high_protein": {
        "family": "lifestyle",
        "strong_keywords": ["high protein", "chicken breast", "salmon", "tofu", "eggs"],
        "weak_keywords": ["chicken", "beef", "turkey", "beans"],
        "negative_keywords": ["candy", "frosting"],
        "threshold": 3,
    },
    "quick": {
        "family": "intent",
        "strong_keywords": ["quick", "15 minute", "20 minute", "under 30", "one pot"],
        "weak_keywords": ["easy", "fast", "simple", "weeknight"],
        "negative_keywords": ["slow cooked", "overnight", "simmer for 45 minutes"],
        "threshold": 2,
    },
}


def _tokenize(text: str) -> str:
    return re.sub(r"\s+", " ", (text or "").lower().strip())


def _keyword_matches(text: str, keyword: str) -> bool:
    normalized_keyword = re.escape(_tokenize(keyword))
    return re.search(rf"(?<![a-z0-9]){normalized_keyword}(?![a-z0-9])", text) is not None


def _count_matches(text: str, keywords: list[str]) -> int:
    return sum(1 for keyword in keywords if _keyword_matches(text, keyword))


def _score_scope(text: str, config: dict) -> int:
    strong_hits = _count_matches(text, config.get("strong_keywords", []))
    weak_hits = _count_matches(text, config.get("weak_keywords", []))
    negative_hits = _count_matches(text, config.get("negative_keywords", []))
    return (strong_hits * 2) + weak_hits - (negative_hits * 2)


def _create_scope(text: str) -> dict[str, list[str]]:
    normalized_text = _tokenize(text)
    scope_tags = {"dish_type": [], "cuisine": [], "intent": [], "lifestyle": [], "audience": []}

    for scope_name, config in FEED_SCOPE_REGISTRY.items():
        family = config["family"]
        score = _score_scope(normalized_text, config)
        if score >= config.get("threshold", 2):
            scope_tags[family].append(scope_name)

    return scope_tags

In [ ]:
examples = [
    "Quick 20 minute chicken breast weeknight bowl with beans",
    "Chocolate brownie dessert with sweet frosting",
    "Slow cooked salmon stew simmer for 45 minutes",
]

for text in examples:
    print(text)
    print(_create_scope(text))
    print()

## Diversity Pass

`_diversify_ranked_posts` runs after scoring.

It assumes each candidate already has:

```text
(post_id, score, normalized_vector)
```

Then it greedily selects the next best candidate after penalizing similarity to already selected posts.

This prevents a feed like:

```text
salmon bowl
salmon rice bowl
salmon quinoa bowl
salmon lunch bowl
```

from dominating the first screen.

In [ ]:
def dot(a, b):
    return sum(x * y for x, y in zip(a, b))


def _diversify_ranked_posts(scored_posts, limit, similarity_penalty, max_similarity, soft_penalty):
    selected = []
    remaining = scored_posts[:]
    result = []

    while remaining and len(result) < limit:
        best_index = 0
        best_score = float("-inf")

        for index, (post_id, base_score, post_vec) in enumerate(remaining):
            if selected:
                nearest_similarity = max(dot(post_vec, selected_vec) for _, selected_vec in selected)
            else:
                nearest_similarity = 0.0

            adjusted_score = base_score

            if nearest_similarity >= max_similarity:
                adjusted_score -= similarity_penalty

            adjusted_score -= nearest_similarity * soft_penalty

            if adjusted_score > best_score:
                best_score = adjusted_score
                best_index = index

        post_id, _, post_vec = remaining.pop(best_index)
        selected.append((post_id, post_vec))
        result.append(post_id)

    return result

In [ ]:
scored_posts = [
    (101, 0.95, [1.0, 0.0, 0.0]),
    (102, 0.94, [0.98, 0.02, 0.0]),
    (103, 0.90, [0.0, 1.0, 0.0]),
    (104, 0.88, [0.0, 0.0, 1.0]),
]

_diversify_ranked_posts(
    scored_posts=scored_posts,
    limit=3,
    similarity_penalty=0.35,
    max_similarity=0.72,
    soft_penalty=0.12,
)

## How Scopes And Diversity Fit Learning

The brain learns relationships from behavior.

Scopes and diversity shape the output:

```text
behavior learning says what is relevant
scope filtering says what category the user asked for
diversity says do not show X amount, we use 5, near-identical things in a row
```

Together, the feed can be both smart and pleasant to browse.